# Clash Royale — Build `cards.csv`

One notebook that rebuilds `data/cards.csv` (and `data/card_attributes.csv`) from scratch.
All the logic lives in [`optimizer/build_dataset.py`](../optimizer/build_dataset.py); this
notebook just runs it step by step and shows what each step found. The same rebuild is
available as `python -m optimizer.build_dataset` or `python main.py --refresh`.

Pipeline: **official API** (id, name, elixir, rarity, evolution) → **Fandom wiki scrape**
(hitpoints, damage, spawn and evolution stats) → **merge** → `cards.csv`.

**Before running:**
1. Provide your API token via the `CR_API_TOKEN` environment variable or a `token.txt` in the
   project root (see `token.txt.example`). Tokens are IP-locked — create one at
   [developer.clashroyale.com](https://developer.clashroyale.com) for your current public IP.
2. Pick the **Python (hackathon)** kernel (needs pandas, requests, lxml).

In [1]:
import json, os, sys
from collections import Counter

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))  # project root (this notebook lives in notebooks/)
from optimizer import config
from optimizer.cr_api import fetch_raw_cards, card_from_api_item, load_cards_csv
from optimizer.build_dataset import (
    build_cards_csv, fetch_wiki_html, parse_wiki_tables, match_wiki_to_api, ALIASES,
)

REFRESH_WIKI = True  # False = reuse data/scrape_cache.html instead of re-downloading

assert config.get_api_token(), "No API token: set CR_API_TOKEN or create token.txt (see token.txt.example)."

## 1. What the API returns
The raw `/cards` response, and which cards are new compared with the current `cards.csv`.

In [2]:
items = fetch_raw_cards()
cards = [card_from_api_item(it) for it in items]
print(f"{len(items)} cards from the API\n")
print("First card, raw:")
print(json.dumps(items[0], indent=2))
print("\nUnion of fields across all cards:", sorted({k for it in items for k in it}))

current = {c.name for c in load_cards_csv()} if config.CARDS_CSV.exists() else set()
names = {c.name for c in cards}
print(f"\nNew since the current cards.csv: {sorted(names - current) or 'none'}")
print(f"Gone from the API:               {sorted(current - names) or 'none'}")

123 cards from the API

First card, raw:
{
  "name": "Knight",
  "id": 26000000,
  "maxLevel": 16,
  "maxEvolutionLevel": 3,
  "elixirCost": 3,
  "iconUrls": {
    "medium": "https://api-assets.clashroyale.com/cards/300/jAj1Q5rclXxU9kVImGqSJxa4wEMfEhvwNQ_4jiGUuqg.png",
    "heroMedium": "https://api-assets.clashroyale.com/cardheroes/300/jAj1Q5rclXxU9kVImGqSJxa4wEMfEhvwNQ_4jiGUuqg.png",
    "evolutionMedium": "https://api-assets.clashroyale.com/cardevolutions/300/jAj1Q5rclXxU9kVImGqSJxa4wEMfEhvwNQ_4jiGUuqg.png"
  },
  "rarity": "common"
}

Union of fields across all cards: ['elixirCost', 'iconUrls', 'id', 'maxEvolutionLevel', 'maxLevel', 'name', 'rarity']

New since the current cards.csv: none
Gone from the API:               none


In [3]:
print("By type:  ", dict(Counter(c.type for c in cards)))
print("By rarity:", dict(Counter(c.rarity for c in cards).most_common()))
print(f"With evolutions: {sum(c.has_evolution for c in cards)}")
print("Champions:      ", ", ".join(c.name for c in cards if c.is_champion))
print("Champion-hero:  ", ", ".join(c.name for c in cards if c.is_champion_hero))

By type:   {'troop': 89, 'building': 13, 'spell': 21}
By rarity: {'epic': 33, 'rare': 31, 'common': 29, 'legendary': 22, 'champion': 8}
With evolutions: 42
Champions:       Mighty Miner, Skeleton King, Archer Queen, Golden Knight, Monk, Little Prince, Goblinstein, Boss Bandit
Champion-hero:   Knight, Goblins, Giant, Balloon, Valkyrie, Musketeer, Wizard, Mini P.E.K.K.A, Ice Wizard, Dark Prince, Bowler, Ice Golem, Mega Minion, Magic Archer, Mighty Miner, Skeleton King, Archer Queen, Golden Knight, Monk, Little Prince, Goblinstein, Berserker, Boss Bandit, Tombstone, Barbarian Barrel


## 2. What the wiki scrape finds
Downloads the Fandom *Cards* page (cached to `data/scrape_cache.html`), keeps every table with
a `Card` column, and coalesces the stat columns per card.

In [4]:
html = fetch_wiki_html(refresh=REFRESH_WIKI)
wiki = parse_wiki_tables(html)
print(f"{wiki.tables_used} of {wiki.tables_found} tables had a Card column -> {len(wiki.frame)} wiki cards")
print("columns:", list(wiki.frame.columns))
wiki.frame.head(10)

    wiki fetch attempt 1/5 failed (HTTP 403)
    wiki fetch attempt 2/5 failed (HTTP 403)
8 of 17 tables had a Card column -> 140 wiki cards
columns: ['card', 'hitpoints', 'damage', 'attack_period', 'damage_per_second', 'special_damage', 'range', 'lifetime', 'crown_tower_damage', 'radius', 'troop_spawned', 'spawn_count_period', 'max_troops_spawned', 'evo_cycles', 'evo_overall_cost', 'evo_stat_boosts']


,card,hitpoints,damage,attack_period,damage_per_second,special_damage,range,lifetime,crown_tower_damage,radius,troop_spawned,spawn_count_period,max_troops_spawned,evo_cycles,evo_overall_cost,evo_stat_boosts
0,Archer Queen,1000.0,225.0,1.2,187.0,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Archers,304.0,112.0,0.9,124.0,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,2.0,9.0,+1 tile Range
2,Arrows,NaN,366.0,NaN,NaN,NaN,NaN,NaN,75.0,3.5,NaN,NaN,NaN,NaN,NaN,NaN
3,Baby Dragon,1152.0,161.0,1.5,107.0,NaN,3.50,NaN,NaN,NaN,NaN,NaN,NaN,2.0,12.0,NaN
4,Balloon,1679.0,640.0,2.0,320.0,240.0,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Bandit,906.0,194.0,1.0,194.0,389.0,0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Barbarian,840.0,96.0,1.0,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Barbarian Barrel,NaN,230.0,NaN,NaN,NaN,NaN,NaN,NaN,4.5,Barbarians,Upon breaking,1,NaN,NaN,NaN
8,Barbarian Hut,1164.0,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,Barbarians,3 every 15 seconds and 1 upon death,8,NaN,NaN,NaN
9,Barbarians,670.0,192.0,1.3,147.0,NaN,0.70,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.0,+10% Hitpoints


In [5]:
attrs, unmatched = match_wiki_to_api(wiki.frame, names)
print(f"{len(attrs)} wiki cards matched to API names")
print(f"\nUnmatched wiki rows ({len(unmatched)}) -- tower troops / spawned sub-units / removed cards are expected:\n  {unmatched}")
print(f"\nAPI cards with no wiki stats ({len(names - set(attrs['name']))}):\n  {sorted(names - set(attrs['name']))}")

118 wiki cards matched to API names

Unmatched wiki rows (22) -- tower troops / spawned sub-units / removed cards are expected:
  ['Barbarian', 'Bush Goblins', 'Cannoneer', 'Cursed Hog', 'Dagger Duchess', 'Elixir Blob', 'Elixir Golemite', 'Goblin', 'Goblin Brawler', 'Golemite', 'Guardienne', 'Lava Pup', 'Monster', 'Phoenix Egg', 'Rascal Boy', 'Rascal Girl', 'Reborn Phoenix', 'Royal Chef', 'Skeleton', 'Skeleton Dragon', 'Spear Goblin', 'Tower Princess']

API cards with no wiki stats (5):
  ['Clone', 'Minion Giant', 'Mirror', 'Rascals', 'Ronin']


If a *real* card shows up in both lists under two spellings, add a `wiki name -> API name`
entry to `ALIASES` in `optimizer/build_dataset.py` and re-run.

## 3. Build `cards.csv`
Runs the whole pipeline (steps 1–2 again, then the merge) and writes both data files. The wiki
page was already downloaded above, so this reuses the cache.

In [6]:
report = build_cards_csv(refresh_scrape=False)

1/3 fetching the card list from the official API ...
    123 cards
2/3 scraping card stats from the wiki ...
    8/17 tables, 140 wiki rows, 118 matched to API cards
3/3 writing data files ...
cards.csv rebuilt: 123 cards x 26 columns -> C:\Users\jerem\Hackathon 2\clash-royale-deck-optimizer\data\cards.csv
  wiki: 8/17 tables used, 140 wiki rows, 118 API cards matched -> card_attributes.csv
  added since last build (0)
  removed since last build (0)
  API cards with no wiki stats (5): Clone, Minion Giant, Mirror, Rascals, Ronin
  unmatched wiki rows (tower troops / sub-units / removed cards are expected) (22): Barbarian, Bush Goblins, Cannoneer, Cursed Hog, Dagger Duchess, Elixir Blob, Elixir Golemite, Goblin, Goblin Brawler, Golemite, Guardienne, Lava Pup, Monster, Phoenix Egg, Rascal Boy, Rascal Girl, Reborn Phoenix, Royal Chef, Skeleton, Skeleton Dragon, Spear Goblin, Tower Princess
  has_evolution disagrees with wiki evolution table (3): Elite Barbarians, Minion Horde, Princess


In [7]:
pd.read_csv(config.CARDS_CSV).sort_values(["type", "rarity", "elixir", "name"]).reset_index(drop=True)

,id,name,elixir,rarity,type,has_evolution,is_champion,is_champion_hero,win_condition,spell_size,...,radius,lifetime,crown_tower_damage,special_damage,troop_spawned,spawn_count_period,max_troops_spawned,evo_cycles,evo_overall_cost,evo_stat_boosts
0,27000000,Cannon,3,common,building,True,False,False,NaN,NaN,...,NaN,30.0,NaN,NaN,NaN,NaN,NaN,2.0,9.0,NaN
1,27000002,Mortar,4,common,building,True,False,False,primary,NaN,...,NaN,30.0,NaN,NaN,NaN,NaN,NaN,2.0,12.0,-1 second Attack Period
2,27000006,Tesla,4,common,building,True,False,False,NaN,NaN,...,NaN,30.0,NaN,NaN,NaN,NaN,NaN,2.0,12.0,NaN
3,27000013,Goblin Drill,4,epic,building,True,False,False,primary,NaN,...,NaN,10.0,NaN,84.0,Goblins,Every 3 seconds and 2 upon death,6,2.0,12.0,NaN
4,27000008,X-Bow,6,epic,building,False,False,False,primary,NaN,...,NaN,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,26000052,Zappies,4,rare,troop,False,False,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,26000003,Giant,5,rare,troop,False,False,True,primary,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,26000059,Royal Hogs,5,rare,troop,True,False,False,primary,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,15.0,NaN
121,26000017,Wizard,5,rare,troop,True,False,True,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.0,NaN
